In [ ]:
"""
This script is used to run pycistarget on the multiome data
authors: Roy Oelen
"""

In [ ]:
%matplotlib inline
# import the libraries
import pycistarget
import pyranges as pr
import os
import glob
# Load cistarget functions
from pycistarget.motif_enrichment_cistarget import *
from pycistarget.motif_enrichment_dem import *
# keep track of the pycistarget version
pycistarget.__version__

In [ ]:
# location of the regions files
path_to_region_sets = '/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/differential_accessibility/limma_dream/output/topics20_otsu_imputed_confined_ncell5/monocyte/merged_regions/'
# get the bed files
region_sets_files = glob.glob(''.join([path_to_region_sets, '*.bed']))
# read the beds
dars = [pr.read_bed(os.path.join(path_to_region_sets, x)) for x in region_sets_files]

In [ ]:
# get the basenames for the files
region_sets_files_basename = [os.path.basename(x) for x in region_sets_files]
region_sets_files_basename = [x.replace('.bed', '') for x in region_sets_files_basename]
region_sets_files_basename = [x.replace('monocyte_', '') for x in region_sets_files_basename]
# make into dictionary
region_sets = {region_sets_files_basename[i]: dars[i] for i in range(len(region_sets_files_basename))}

In [ ]:
# get as pandas dataframe
region_sets_pd_list = [pyr_object.as_df() for pyr_object in region_sets.values()]
# merge them
region_sets_pd_all = pd.concat(region_sets_pd_list)
# change columns to be in line with bed file annotations
region_sets_pd_all.columns = ['#chrom', 'chromStart', 'chromEnd', 'name']
# write result
bed_merged_loc = ''.join(['/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/differential_accessibility/limma_dream/output/topics20_otsu_imputed_confined_ncell5/monocyte/merged_regions/all_topics/', 'all_topics.bed'])
region_sets_pd_all.to_csv(bed_merged_loc, sep = '\t', header = True, index = False)

In [5]:
# set parameters
ctx_db = '/groups/umcg-franke-scrna/tmp04/external_datasets/cistarget/hg38_screen_v10_clust.regions_vs_motifs.rankings.feather'
specie = 'homo_sapiens'
path_to_motif_annotations = '/groups/umcg-franke-scrna/tmp04/external_datasets/cistarget/motifs-v10nr_clust-nr.hgnc-m0.001-o0.0.tbl'
_temp_dir = '/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/tmp_space/'
pycistarget_output_dir = '/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/limma_pycistarget/'

In [6]:
# get command
pycistarget_command = ' '.join(['pycistarget cistarget', 
                                '--cistarget_db_fname', ctx_db, 
                                '--bed_fname', bed_merged_loc , 
                                '--output_folder', pycistarget_output_dir ,
                                '--species', specie, 
                                '--write_html'])
# run command
os.system(pycistarget_command)

2024-11-25 15:24:59,674 cisTarget    INFO     Reading cisTarget database
2024-11-25 15:26:49,633 cisTarget    INFO     Running cisTarget for all_topics which has 217616 regions
2024-11-25 15:27:24,097 cisTarget    INFO     Annotating motifs for all_topics
2024-11-25 15:27:31,020 cisTarget    INFO     Getting cistromes for all_topics


0

In [7]:
# make topic and location of bed into dictionary
region_sets_paths = {region_sets_files_basename[i]: region_sets_files[i] for i in range(len(region_sets_files_basename))}
# also try each topic separately
for key, value in region_sets_paths.items():
    # get the output directory
    out_dir_topic = ''.join([pycistarget_output_dir, '/', key, '/'])
    if not os.path.exists(out_dir_topic):
        os.makedirs(out_dir_topic)
    # create the command
    pycistarget_command_topic = ' '.join(['pycistarget cistarget', 
                                '--cistarget_db_fname', ctx_db, 
                                '--bed_fname', value , 
                                '--output_folder',  out_dir_topic,
                                '--species', specie, 
                                '--write_html'])
    # run command
    os.system(pycistarget_command_topic)

2024-11-25 15:28:06,936 cisTarget    INFO     Reading cisTarget database
2024-11-25 15:28:29,326 cisTarget    INFO     Running cisTarget for monocyte_Topic12 which has 19971 regions
2024-11-25 15:28:44,855 cisTarget    INFO     Annotating motifs for monocyte_Topic12
2024-11-25 15:28:47,815 cisTarget    INFO     Getting cistromes for monocyte_Topic12
2024-11-25 15:28:56,105 cisTarget    INFO     Reading cisTarget database
2024-11-25 15:29:17,197 cisTarget    INFO     Running cisTarget for monocyte_Topic2 which has 13167 regions
2024-11-25 15:29:31,140 cisTarget    INFO     Annotating motifs for monocyte_Topic2
2024-11-25 15:29:33,976 cisTarget    INFO     Getting cistromes for monocyte_Topic2
2024-11-25 15:29:40,748 cisTarget    INFO     Reading cisTarget database
2024-11-25 15:30:02,054 cisTarget    INFO     Running cisTarget for monocyte_Topic9 which has 15552 regions
2024-11-25 15:30:16,938 cisTarget    INFO     Annotating motifs for monocyte_Topic9
2024-11-25 15:30:20,094 cisTarget 

In [8]:
# Run, using precomputed database. This was the old method that uses python commands instead of a shell command
# cistarget_dict = run_pycistarget(ctx_db = ctx_db,
#                                                       region_sets = region_sets,
#                                                       specie = specie,
#                                                       auc_threshold = 0.005,
#                                                       nes_threshold = 3.0,
#                                                       rank_threshold = 0.05,
#                                                       annotation = ['Direct_annot', 'Orthology_annot'],
#                                                       annotation_version = 'v10nr_clust',
#                                                       path_to_motif_annotations = path_to_motif_annotations,
#                                                       n_cpu = 4,
#                                                       _temp_dir=_temp_dir)

In [ ]:
dem_db = '/groups/umcg-franke-scrna/tmp04/external_datasets/cistarget/hg38_screen_v10_clust.regions_vs_motifs.scores.feather'
pycistarget_dem_dir = '/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/limma_pycistarget/dems/'
background_beds = ['/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/signac_peaks/output/mo_peaks_lane1to80_monocyte_exp0001.bed']

In [ ]:
# get command
pycistarget_command = ' '.join(['pycistarget dem', '--dem_db_fname',  dem_db,
                                '--foreground_beds', ' '.join(region_sets_files), 
                                '--background_beds', ' '.join(background_beds), 
                                #'--background_beds', ' '.join(region_sets_files), 
                                '--output_folder', pycistarget_dem_dir, 
                                '--species', specie, 
                                '--genome_annotation', path_to_motif_annotations,
                                '--seed', '1337', 
                                '--write_html'])
# run command
os.system(pycistarget_command)

In [11]:
# DEM_dict = DEM(dem_db = dem_db,
#     region_sets = region_sets,
#     specie = specie,
#     contrasts = 'Other',
#     name = 'DEM',
#     fraction_overlap = 0.4,
#     max_bg_regions = 500,
#     adjpval_thr = 0.05,
#     log2fc_thr = 1,
#     mean_fg_thr = 0,
#     motif_hit_thr = None,
#     cluster_buster_path = None,
#     path_to_genome_fasta = None,
#     path_to_motifs = None,
#     annotation_version = 'v10nr_clust',
#     path_to_motif_annotations = path_to_motif_annotations,
#     motif_annotation = ['Direct_annot', 'Orthology_annot'],
#     n_cpu = 4,
#     tmp_dir = _temp_dir,
#     _temp_dir=_temp_dir)